# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR2) Exploration with `mlcroissant`
This notebook provides an example workflow for loading, exploring, and analyzing a Croissant/FAIR dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets (by their `@id`) and summarize their fields and columns. All entities referenced by their `@id`.

Below, we print all record set `@id`s, and for each, the associated fields' and columns' `@id`s and names.

In [ ]:
# List all record sets by @id and describe their structure
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}    name: {field.name}    dataType: {field.data_type if hasattr(field, 'data_type') else ''}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - @id: {col.id}    name: {col.name}")
    print('-' * 60)

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.
Each record set is referenced by its `@id`.

In [ ]:
# List record set @ids for extraction
record_sets = dataset.record_sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"{len(df)} records loaded for {record_set_id}.")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Apply standard preprocessing and analysis steps. Here, we select a numeric field (`@id`) for demonstration, filter records, normalize, and group by a categorical attribute. All references are by `@id`.

**Modify the variables below to reference relevant record set, numeric field, and group field `@id` from the previous cell's output.**

In [ ]:
# --- Adjust these variables based on dataset structure ---
record_set_id = next(iter(dataframes))  # Select first available record set for example
df = dataframes[record_set_id]

# Review columns to pick candidates
print(f"Columns in {record_set_id}:")
print(df.columns.tolist())

# Example: suppose one numeric field is 'age' with @id='age' and group field is 'sex' with @id='sex'
# Replace these with the actual @id strings from your overview as needed
numeric_field_id = 'age'
group_field_id = 'sex'

if numeric_field_id in df.columns:
    # Filtering records
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())
    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field '{numeric_field_id}' not found in columns. Please update 'numeric_field_id' with a valid column @id.")

## 5. Visualization
Visualize data distributions or relationships. Here is an example histogram for the numeric field and a boxplot grouped by a categorical field, using the selected `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure variable from previous cell
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=15)
    plt.title(f'Histogram of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading a FAIR-structured medical dataset via its Croissant schema, inspecting the schema and record structure, extracting data by `@id`, and performing basic exploratory analysis and visualization. You may now proceed to further statistical analysis or modeling based on these principles.

**Tip:** For customized analyses, always reference record sets and fields by their unique `@id`.